<a href="https://colab.research.google.com/github/KawiraK/_Web_Scraping/blob/main/week_1_prac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 Practical: Scraping a Live Website with Python

**Course:** Data & AI Specialist Program, Cyber Shujaa
**Where this fits:** In Week 1 we outlined the Data Science Methodology. This notebook is a live,
worked example of the **Data Understanding** step: going out and getting real data ourselves,
instead of being handed a clean file to work with.

**What we are building:** a small Python pipeline that visits a real website, reads its HTML,
pulls out one table, and turns it into a pandas DataFrame we can actually work with.

**Our target:** [scrapethissite.com/pages/forms/](https://www.scrapethissite.com/pages/forms/): a
site built specifically for scraping practice. It lists NHL hockey team statistics (wins, losses,
goals for/against, etc.) in one HTML `<table>`, one row per team per year. It is safe and legal to
scrape because the site exists for exactly this purpose.

**Why this matters beyond this one table:** the four-step pattern here (fetch the page, parse it,
find the repeating structure, loop through it into a DataFrame) is the same pattern you will reuse
on almost any scraping task, no matter what site or what data. Once this clicks, Assignment #1
is the same four steps applied to a website of your own choosing.


## 1. Bring in the tools

In [ ]:
# Three libraries, three jobs in this pipeline:
from bs4 import BeautifulSoup   # turns raw HTML text into something we can search
import requests                 # fetches pages over the internet (HTTP)
import pandas as pd             # stores and organizes the data we extract


`requests`: This is our "messenger". It handles the job of going to the internet, visiting a specific URL, and bringing back whatever is on that page.

`BeautifulSoup`: This is our "translator". Websites are written in HTML, which looks like a mess of tags. BeautifulSoup helps us navigate those tags to find the text we want.

`pandas`: This is our "data organizer". It allows us to put our data into a DataFrame, which is essentially a powerful digital spreadsheet.

## 2. Visiting the Website

In [ ]:
url = 'https://www.scrapethissite.com/pages/forms/'   # the page we want data from

page = requests.get(url)                # send the request, get the response back
soup = BeautifulSoup(page.text, 'html.parser')  # hand the raw HTML text to our translator


`requests.get(url)`: This line tells our messenger to go to the hockey site and grab the data.

`page.text`: This is the raw "Source Code" of the page.

`BeautifulSoup(..., 'html.parser')`: We feed that raw text into the translator so we can start searching it.

## 3. Finding the Table and Headers

In [ ]:
hockey_table = soup.find('table', class_='table')   # locate the one <table> we care about

table_titles = hockey_table.find_all('th')          # every column header (<th>) inside it
hockey_table_title = [title.text.strip() for title in table_titles]  # clean header text, e.g. "Team Name"

df = pd.DataFrame(columns=hockey_table_title)        # empty spreadsheet, headers already in place


`find('table')`: We tell the translator to look for the <table> tag.

`find_all('th')`: We look for the "Table Headers". These are the labels like "Team Name" or "Wins."

`.text.strip()`: This is a cleanup step. It grabs the text and "strips" away any invisible extra spaces.

`pd.DataFrame(columns=...)`: We create an empty spreadsheet and use those headers as our top row.

## 4. Collecting the Rows

In [ ]:
table_data = hockey_table.find_all('tr')   # every table row (<tr>), including the header row

for row in table_data[1:]:                 # skip index 0 - that row is the headers, not data
  raw_data = row.find_all('td')            # the individual data cells (<td>) in this row
  each_raw_data = [data.text.strip() for data in raw_data]  # clean text out of each cell

  length = len(df)          # how many rows df has right now = the next free row number
  df.loc[length] = each_raw_data   # paste this row's data into that next free row


`find_all('tr')`: This finds every single row in the table.

The Loop `(for row in...)`: We go through the rows one by one.

`row.find_all('td')`: For each row, we grab the individual data cells (<td>).

`df.loc[length]`: This takes that row of data and "pastes" it into the next available line in our spreadsheet.

## 5. Checking Our Work

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")   # a quick sanity check on size
df.head()   # look at the first 5 rows - do they look like real team stats, not blank/garbled?


Rows: 25, Columns: 9


,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25


Always look before you trust the result. `df.shape` tells us how many rows and columns we ended
up with: if it is `(0, 9)` or similar, something upstream silently found nothing (a wrong class
name is the usual culprit). `df.head()` shows the first five rows so we can eyeball that the data
actually looks like team names, win counts, and years, not empty strings or `None`.

## 6. Saving Our Work

In [ ]:
output_filename = 'hockey_teams.csv'   # a clear, descriptive filename

df.to_csv(output_filename, index=False)   # write the DataFrame out to a real file
# index=False: we don't want pandas' own row-number column saved as an extra column in the file

check_df = pd.read_csv(output_filename)   # read it back in, to prove the save actually worked
print(f"Saved and reloaded {check_df.shape[0]} rows, {check_df.shape[1]} columns.")
check_df.head()


Saved and reloaded 25 rows, 9 columns.


,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,NaN,0.550,299,264,35
1,Buffalo Sabres,1990,31,30,NaN,0.388,292,278,14
2,Calgary Flames,1990,46,26,NaN,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,NaN,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,NaN,0.425,273,298,-25


`df.to_csv(...)`: A DataFrame only exists in memory while this notebook is running. Exporting to
`.csv` (Comma-Separated Values) turns it into a real, portable file, one you can reopen in Excel,
email to someone, or load back into a different notebook entirely.

`index=False`: without this, pandas adds its own row-number column (0, 1, 2, ...) into the saved
file as if it were real data, which it isn't.

`pd.read_csv(...)` right after: this isn't strictly required, but it's good practice. Rather than
just trusting that `to_csv()` didn't raise an error, we reload the file and check its shape as proof
the file exists, is readable, and has the row/column count we expect.

## Wrap-Up: What We Just Did

We walked through the same four moves you will reuse on almost every scraping task:

1. **Fetch**: `requests.get()` retrieved the raw HTML of a live page.
2. **Parse**: `BeautifulSoup` turned that raw HTML into something searchable.
3. **Find the repeating structure**: `find()` and `find_all()` located the one table, its
   headers, and then every row inside it.
4. **Collect into a DataFrame**: a loop pulled the text out of each cell and built a proper,
   analysis-ready table, which we then saved to `.csv` so the work doesn't disappear when this
   notebook closes.

Mapped back to the Data Science Methodology from class: everything above is the **Data
Understanding** step (going out and collecting real data) plus a first light pass at **Data
Preparation** (cleaning stray whitespace with `.strip()` before it ever reaches the DataFrame).

**Next up, Assignment #1:** you will repeat this exact four-step pattern on a website you choose
yourself (not this hockey site), and take Data Preparation further: checking for missing values,
duplicate rows, and columns with the wrong data type before exporting your own `.csv`. Open
`Week1_Assignment1_Web_Scraping_Starter.ipynb` to get started.
